# Kaggriculture Platform Smoke Test

Purpose: verify Kaggle's actual kernel runtime can (a) import the
`kaggriculture` environment and (b) run a packaged agent through a full
season, **before** any behavioral-cloning or PPO work claims Kaggle GPU
progress. This is platform-compatibility verification only — it does not
validate the agent's strategy, and a pass here does not authorize a
competition submission (`kaggle competitions submit` is a separate,
explicit action).

Background: `kaggriculture` is not in the latest published
`kaggle-environments` PyPI release as of 2026-08-01 (this repo installs it
from GitHub `master` locally) — so whether Kaggle's own kernel image has a
compatible build is a genuine open question this notebook answers
empirically, not a formality.

See `docs/superpowers/specs/2026-08-01-kaggriculture-competition-plan-design.md`
§9 ("Codex Kaggle execution-status audit") for the full state vocabulary
this notebook's result should be reported against.

## 1. Environment fingerprint

In [ ]:
import json
import platform
import sys

env_info = {
    "python_version": sys.version,
    "platform": platform.platform(),
}

try:
    import kaggle_environments
    env_info["kaggle_environments_version"] = getattr(kaggle_environments, "__version__", "unknown")
except ImportError as e:
    env_info["kaggle_environments_version"] = None
    env_info["kaggle_environments_import_error"] = repr(e)

try:
    import torch
    env_info["torch_version"] = torch.__version__
    env_info["cuda_available"] = torch.cuda.is_available()
    env_info["cuda_version"] = torch.version.cuda
    env_info["gpu_name"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
except ImportError as e:
    env_info["torch_version"] = None
    env_info["torch_import_error"] = repr(e)

print(json.dumps(env_info, indent=2))

## 2. Import kaggriculture (no explicit install attempted first)

Kaggle's own competition-evaluation backend has to run `kaggriculture` for
the ladder to function, so the most likely outcome is this kernel's image
already has a compatible `kaggle-environments` build. Only if this fails
does the offline-safe fallback (a locally-built wheel attached as a Kaggle
Dataset input, installed via `--no-index`) become necessary — not built
speculatively ahead of that evidence.

In [ ]:
from kaggle_environments import make

kaggriculture_import_ok = False
kaggriculture_import_error = None
try:
    _probe_env = make("kaggriculture", configuration={"episodeSteps": 24}, debug=True)
    kaggriculture_import_ok = True
    print("kaggriculture environment registered and instantiable.")
except Exception as e:
    kaggriculture_import_error = repr(e)
    print(f"FAILED to instantiate kaggriculture: {e!r}")
    raise

## 3. Write out the packaged `roi_teacher_v3` artifact

Embedded verbatim from `build/roi_teacher_v3/main.py` (generated by
`scripts/package_agent.py` — inlines `economy.py` as an in-memory module,
so this file has no dependency on this repo's `src/` layout, matching
exactly what a real `kaggle competitions submit` would receive). Hashed so
the remote hash can be compared against the local build hash recorded in
this project's docs.

In [ ]:
PACKAGED_AGENT_SOURCE = '"""Auto-generated self-contained submission artifact. Do not edit by\nhand — edit roi_teacher_v3/main.py and/or src/kaggriculture_lib, then rerun\nscripts/package_agent.py to regenerate."""\n\nfrom __future__ import annotations\n\nimport types as _types\n\n_economy_src = \'"""Kaggriculture game economy: prices, yields, and derived ROI estimates.\\n\\nEvery formula here mirrors the environment\\\'s own implementation exactly\\n(installed at `kaggle_environments/envs/kaggriculture/kaggriculture.py`, see\\n`docs/2_environment_notes.md` for the version pin and line-range citations\\nper formula). This module exists so every agent version (heuristic, BC,\\nPPO) shares one tested source of truth instead of re-deriving the game math\\nindependently — see `docs/0_coding_standards.md` §2.\\n"""\\n\\nfrom __future__ import annotations\\n\\nimport math\\nfrom typing import Literal\\n\\nShapeFn = Literal["linear", "sq", "sqrt", "log", "log10"]\\n\\nMARKET_I0 = 10_000\\nPRICE_FLOOR = 1\\n\\n# Mirrors kaggriculture.py:40-50 (MARKET_PARAMS) verbatim.\\nMARKET_PARAMS: dict[str, dict] = {\\n    "WHEAT":      {"base":  25, "I0": MARKET_I0, "T": 400, "below_func": "sqrt",   "below_target": 0.80, "above_func": "log",  "above_target": 0.20},\\n    "CARROT":     {"base":  35, "I0": MARKET_I0, "T": 450, "below_func": "log",    "below_target": 0.20, "above_func": "sqrt", "above_target": 0.70},\\n    "TOMATO":     {"base":  60, "I0": MARKET_I0, "T": 200, "below_func": "linear", "below_target": 0.40, "above_func": "sqrt", "above_target": 0.60},\\n    "STRAWBERRY": {"base": 120, "I0": MARKET_I0, "T": 100, "below_func": "sqrt",   "below_target": 0.70, "above_func": "linear", "above_target": 1.60},\\n    "MELON":      {"base": 250, "I0": MARKET_I0, "T": 300, "below_func": "log",    "below_target": 0.20, "above_func": "sq",   "above_target": 3.60},\\n    "EGG":        {"base":  50, "I0": MARKET_I0, "T": 332, "below_func": "linear", "below_target": 0.40, "above_func": "log",  "above_target": 0.20},\\n    "MILK":       {"base": 160, "I0": MARKET_I0, "T": 122, "below_func": "sqrt",   "below_target": 0.60, "above_func": "linear", "above_target": 1.60},\\n    "WOOL":       {"base": 200, "I0": MARKET_I0, "T": 105, "below_func": "log",    "below_target": 0.20, "above_func": "sq",   "above_target": 3.20},\\n    "FERTILIZER": {"base": 100, "I0": MARKET_I0, "T": 200, "below_func": "linear", "below_target": 0.40, "above_func": "linear", "above_target": 0.40},\\n}\\n\\n# Mirrors kaggriculture.py:11-17 (CROPS) verbatim.\\nCROPS: dict[str, dict] = {\\n    "WHEAT":      {"seed": 10, "first_yield_day": 2, "max_yield_day": 4, "interval": 0, "max_yield": 6, "ongoing": False},\\n    "CARROT":     {"seed": 20, "first_yield_day": 2, "max_yield_day": 3, "interval": 0, "max_yield": 4, "ongoing": False},\\n    "TOMATO":     {"seed": 50, "first_yield_day": 8, "max_yield_day": 8, "interval": 1, "max_yield": 4, "ongoing": True},\\n    "STRAWBERRY": {"seed": 100, "first_yield_day": 10, "max_yield_day": 10, "interval": 2, "max_yield": 4, "ongoing": True},\\n    "MELON":      {"seed": 80, "first_yield_day": 10, "max_yield_day": 12, "interval": 0, "max_yield": 6, "ongoing": False},\\n}\\n\\n# Mirrors kaggriculture.py:19-23 (ANIMALS) verbatim.\\nANIMALS: dict[str, dict] = {\\n    "GOOSE": {"cost": 300, "structure": "COOP", "first_yield_day": 4, "interval": 1, "max_held": 4, "product": "EGG"},\\n    "COW":   {"cost": 400, "structure": "PASTURE", "first_yield_day": 8, "interval": 2, "max_held": 6, "product": "MILK"},\\n    "SHEEP": {"cost": 500, "structure": "PASTURE", "first_yield_day": 6, "interval": 3, "max_held": 6, "product": "WOOL"},\\n}\\n\\n# Mirrors kaggriculture.py:82-83 (LAND_ORDER / LAND_PRICES) verbatim.\\nLAND_ORDER = ["NE", "SW", "SE"]\\nLAND_PRICES = [1000, 2000, 4000]\\n\\nFARM_HAND_COST_MULT = 1\\n\\n\\ndef _shape(func: ShapeFn, x: float) -> float:\\n    """Mirrors kaggriculture.py:53-60 (`_shape`)."""\\n    x = max(0.0, x)\\n    if func == "linear":\\n        return x\\n    if func == "sq":\\n        return x * x\\n    if func == "sqrt":\\n        return math.sqrt(x)\\n    if func == "log":\\n        return math.log(1.0 + x)\\n    if func == "log10":\\n        return math.log10(1.0 + x)\\n    return x\\n\\n\\ndef market_price(item: str, inventory: float, params: dict | None = None) -> int:\\n    """Current sale price for `item` at the given market inventory level.\\n\\n    Mirrors kaggriculture.py:177-191 (`market_price`) exactly:\\n    `price(inv) = base + sign*amp*f(|inv-I0|)`, floored at `PRICE_FLOOR` and\\n    rounded to the nearest int. `sign` is +1 (scarcity) below I0, -1 (glut)\\n    above it; `amp` is derived so that moving `T` units past `I0` shifts\\n    price by `target * base`.\\n    """\\n    p = (params or MARKET_PARAMS)[item]\\n    base, i0, t = p["base"], p["I0"], p["T"]\\n    if inventory < i0:\\n        f = p["below_func"]\\n        amp = p["below_target"] * base / _shape(f, t)\\n        price = base + amp * _shape(f, i0 - inventory)\\n    else:\\n        f = p["above_func"]\\n        amp = p["above_target"] * base / _shape(f, t)\\n        price = base - amp * _shape(f, inventory - i0)\\n    return max(PRICE_FLOOR, round(price))\\n\\n\\ndef hire_cost(n_already_today: int, mult: int = FARM_HAND_COST_MULT) -> int:\\n    """Cost of the next hire today. Mirrors kaggriculture.py:658-667.\\n\\n    `_fib(n)` is indexed so `_fib(0)=1, _fib(1)=1, _fib(2)=2, _fib(3)=3, ...`\\n    and resets to 0 at the start of each day (`farm["hires_today"]`).\\n    """\\n    a, b = 1, 1\\n    for _ in range(n_already_today):\\n        a, b = b, a + b\\n    return mult * a\\n\\n\\ndef land_cost(n_unlocked_extra: int) -> int | None:\\n    """Cost of the next `BUY_LAND` order, or None if all land is unlocked.\\n\\n    Mirrors kaggriculture.py:680-693 (`_do_buy_land`). `n_unlocked_extra` is\\n    the count of quadrants already bought beyond the always-unlocked NW.\\n    """\\n    if n_unlocked_extra >= len(LAND_ORDER):\\n        return None\\n    return LAND_PRICES[n_unlocked_extra]\\n\\n\\ndef one_time_crop_watering_bonus_window(crop: str) -> tuple[int, int]:\\n    """Inclusive (start, end) age-in-days window where watering adds yield.\\n\\n    Mirrors kaggriculture.py:373-386 (`WATER` handler): window starts at\\n    `(max_yield_day + 1) // 2` (== ceil(max_yield_day / 2)) through\\n    `max_yield_day` inclusive. Only meaningful for non-ongoing crops.\\n    """\\n    cd = CROPS[crop]\\n    if cd["ongoing"]:\\n        raise ValueError(f"{crop} is an ongoing-yield crop, not one-time")\\n    start = (cd["max_yield_day"] + 1) // 2\\n    return start, cd["max_yield_day"]\\n\\n\\ndef ongoing_crop_production_days(crop: str) -> list[int]:\\n    """Days-since-planting (0-indexed) on which an ongoing crop ticks yield.\\n\\n    Mirrors kaggriculture.py:738-771 (`_daily_refresh_plants`): production\\n    ticks when `(day_since_planting - first_yield_day) % interval == 0`,\\n    for up to `max_yield` ticks.\\n    """\\n    cd = CROPS[crop]\\n    if not cd["ongoing"]:\\n        raise ValueError(f"{crop} is a one-time-yield crop, not ongoing")\\n    days = []\\n    tick = 0\\n    day = cd["first_yield_day"]\\n    while tick < cd["max_yield"]:\\n        days.append(day)\\n        tick += 1\\n        day += cd["interval"]\\n    return days\\n\\n\\ndef animal_production_days(animal: str) -> list[int]:\\n    """Days-since-placement (0-indexed) on which an animal ticks a base yield.\\n\\n    Mirrors kaggriculture.py:774-802 (`_daily_refresh_animals`). Does not\\n    include CARE-bonus timing, which depends on per-day feed/care history\\n    rather than a fixed schedule.\\n    """\\n    a = ANIMALS[animal]\\n    days = []\\n    tick = 0\\n    day = a["first_yield_day"]\\n    while tick < a["max_held"]:\\n        days.append(day)\\n        tick += 1\\n        day += a["interval"]\\n    return days\\n\\n\\ndef one_time_crop_static_yield_per_tile_day(crop: str, watered_in_window: bool = True) -> float:\\n    """Simple static estimate of average yield/tile/day over the crop\\\'s life.\\n\\n    Not a substitute for simulating an actual play line — ignores fertilizer,\\n    weeds, and opportunity cost of the farmer\\\'s actions to water/harvest.\\n    Intended as a first-pass ranking signal for `docs/3_agent_strategy.md`,\\n    matching the design doc\\\'s Phase-1 "static $/tile/day tables" deliverable.\\n    """\\n    cd = CROPS[crop]\\n    base_units = 1  # tile always yields >= 1 unit on harvest (see kaggriculture.py:200-213)\\n    if watered_in_window:\\n        start, end = one_time_crop_watering_bonus_window(crop)\\n        bonus_days = end - start + 1\\n        base_units += bonus_days  # +1 unit per watered day in the bonus window\\n    base_units = min(base_units, cd["max_yield"])\\n    lifespan_days = cd["max_yield_day"] + 1  # decay begins one day after max_yield_day\\n    return base_units / lifespan_days\\n\'\neconomy = _types.ModuleType("economy")\nexec(compile(_economy_src, "kaggriculture_lib/economy.py", "exec"), economy.__dict__)\n\n\n"""Kaggriculture ROI-heuristic teacher agent, v3.\n\nOne-variable change from v2 (`agents/roi_teacher_v2`), per\ndocs/0_coding_standards.md §4: adds a season-horizon gate so the agent\nnever plants a crop that cannot fully mature before the episode ends.\n\nFixes a real gap Codex\'s code review found in v1/v2 (see the design doc\'s\n2026-08-01 "Codex review of Claude\'s current implementation" entry):\nneither version checked remaining season length before planting, so a\nlate-season Melon purchase could spend money that never converts back to\nbank balance before the episode ends (unsold/unharvested assets don\'t\ncount at termination). `kaggle_environments` passes `(observation,\nconfiguration)` and truncates to the agent function\'s actual arg count\n(verified in `kaggle_environments/agent.py`), so accepting `config` here\ngives the real `episodeSteps`/`turnsPerDay` rather than a hardcoded guess.\n\nStill single farmer, single tile (the spawn tile — no movement), no hands,\nno land purchases, no animals, no ongoing crops. See v1\'s docstring for\nthe full scope rationale; unchanged here.\n\nLocal testing only: imports `kaggriculture_lib.economy` assuming `src/` is\non `sys.path` (handled by `scripts/run_tournament.py`). Use\n`scripts/package_agent.py` to generate a standalone submission artifact.\n"""\n\n\n\nCANDIDATE_CROPS = ("WHEAT", "CARROT", "MELON")\n\nDEFAULT_EPISODE_STEPS = 720\nDEFAULT_TURNS_PER_DAY = 24\n\n\ndef _expected_total_units(crop: str) -> int:\n    """Total harvestable units assuming watering every day in the bonus window."""\n    cd = economy.CROPS[crop]\n    start, end = economy.one_time_crop_watering_bonus_window(crop)\n    bonus_days = end - start + 1\n    return min(cd["max_yield"], 1 + bonus_days)\n\n\ndef _score_crop(crop: str, price: float) -> float:\n    """Static $/day ROI estimate: (expected revenue - seed cost) / lifespan."""\n    cd = economy.CROPS[crop]\n    lifespan_days = cd["max_yield_day"] + 1\n    revenue = _expected_total_units(crop) * price\n    return (revenue - cd["seed"]) / lifespan_days\n\n\ndef _last_day_index(config) -> int:\n    """0-indexed final day of the season, from the real episode config."""\n    episode_steps = config.get("episodeSteps", DEFAULT_EPISODE_STEPS) if config else DEFAULT_EPISODE_STEPS\n    turns_per_day = config.get("turnsPerDay", DEFAULT_TURNS_PER_DAY) if config else DEFAULT_TURNS_PER_DAY\n    season_days = episode_steps // turns_per_day\n    return season_days - 1\n\n\ndef _can_mature_in_time(crop: str, current_day: int, last_day_index: int) -> bool:\n    """True iff a crop planted today reaches `max_yield_day` age on or\n    before the season\'s last day, leaving turns that day to harvest."""\n    return current_day + economy.CROPS[crop]["max_yield_day"] <= last_day_index\n\n\ndef _feasible_crops(current_day: int, last_day_index: int) -> tuple[str, ...]:\n    return tuple(c for c in CANDIDATE_CROPS if _can_mature_in_time(c, current_day, last_day_index))\n\n\ndef _best_crop(market_prices: dict[str, int], feasible: tuple[str, ...]) -> str | None:\n    if not feasible:\n        return None\n    return max(\n        feasible,\n        key=lambda crop: _score_crop(crop, market_prices.get(crop, economy.CROPS[crop]["seed"])),\n    )\n\n\ndef agent(obs, config=None):\n    player = obs["player"]\n    me = obs["farms"][player]\n    private = obs["private"]\n    prices = obs["market"]["prices"]\n    shed = private["shed"]\n    farmer_inventory = private["inventories"][0] if private["inventories"] else {}\n\n    fx, fy = me["farmer"]\n    tile = me["tiles"][fy][fx]\n    day = obs["day"]\n\n    market_orders = [\n        ["SELL", crop, shed[crop]] for crop in CANDIDATE_CROPS if shed.get(crop, 0) > 0\n    ]\n\n    farmer_action = ["PASS"]\n\n    if isinstance(tile, dict) and tile.get("kind") == "PLANT":\n        cd = economy.CROPS[tile["crop"]]\n        if not tile["watered_today"]:\n            farmer_action = ["WATER"]\n        elif day - tile["planted_day"] >= cd["max_yield_day"]:\n            farmer_action = ["HARVEST"]\n    elif isinstance(tile, dict) and tile.get("kind") == "WEED":\n        farmer_action = ["DIG"]\n    elif tile is None:\n        # Push any just-harvested produce to the shed before replanting —\n        # the spawn tile is always shed-adjacent (kaggriculture.py\'s\n        # `_default_spawn` picks a shed-access tile), so DROP always lands.\n        if any(farmer_inventory.get(crop, 0) > 0 for crop in CANDIDATE_CROPS):\n            farmer_action = ["DROP"]\n        else:\n            last_day_index = _last_day_index(config)\n            feasible = _feasible_crops(day, last_day_index)\n            crop = _best_crop(prices, feasible)\n            if crop is not None:\n                if private["seeds"].get(crop, 0) > 0:\n                    farmer_action = ["PLANT", crop]\n                elif me["money"] >= economy.CROPS[crop]["seed"]:\n                    market_orders.append(["BUY_SEED", crop, 1])\n            # else: no candidate crop can mature before season end — hold\n            # (PASS), never spend on a seed that can\'t come back as money.\n\n    return {"farmer": farmer_action, "hands": [], "market": market_orders}\n\n'

import hashlib
from pathlib import Path

agent_path = Path("/kaggle/working/roi_teacher_v3_main.py")
agent_path.write_text(PACKAGED_AGENT_SOURCE)
sha256 = hashlib.sha256(PACKAGED_AGENT_SOURCE.encode()).hexdigest()
print(f"Wrote {agent_path} ({len(PACKAGED_AGENT_SOURCE)} bytes)")
print(f"SHA-256: {sha256}")

## 4. Full paired-seat match against `starter`

Same recorded seed, both seat assignments — confirms the packaged agent
runs to completion (`DONE` status, finite reward) on Kaggle's actual
runtime, not just this repo's local `.venv`.

In [ ]:
import math

SMOKE_SEED = 20260801
EPISODE_STEPS = 720

def run_and_validate(agents):
    env = make("kaggriculture", configuration={"episodeSteps": EPISODE_STEPS, "seed": SMOKE_SEED}, debug=True)
    env.run(agents)
    final = env.steps[-1]
    statuses = [s.status for s in final]
    rewards = [s.reward for s in final]
    assert all(s == "DONE" for s in statuses), f"non-DONE status: {statuses}"
    assert all(r is not None and math.isfinite(r) for r in rewards), f"invalid reward: {rewards}"
    return statuses, rewards

seat_a_statuses, seat_a_rewards = run_and_validate([str(agent_path), "starter"])
print(f"Seat A (agent=player0): statuses={seat_a_statuses} rewards={seat_a_rewards}")

seat_b_statuses, seat_b_rewards = run_and_validate(["starter", str(agent_path)])
print(f"Seat B (agent=player1): statuses={seat_b_statuses} rewards={seat_b_rewards}")

## 5. Write machine-readable result artifact

In [ ]:
import datetime

result = {
    "status": "kernel_complete",
    "generated_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "environment": env_info,
    "kaggriculture_import_ok": kaggriculture_import_ok,
    "kaggriculture_import_error": kaggriculture_import_error,
    "packaged_agent_sha256": sha256,
    "episode_steps": EPISODE_STEPS,
    "seed": SMOKE_SEED,
    "seat_a_agent0_vs_starter1": {"statuses": seat_a_statuses, "rewards": seat_a_rewards},
    "seat_b_starter0_vs_agent1": {"statuses": seat_b_statuses, "rewards": seat_b_rewards},
}

with open("/kaggle/working/smoke_result.json", "w") as f:
    json.dump(result, f, indent=2)

print(json.dumps(result, indent=2))
print("\nSMOKE TEST PASSED")